# 02 Head-to-Full Long DQA

This notebook tests the new FedSTO-style DQA schedule:

- **Phase 1:** long head/neck-only DQA adaptation, default 30 rounds.
- **Phase 2:** short full-model low-LR DQA burst, default 2 rounds.
- **Evaluation:** final-focused paper-protocol evaluation only, so the
  long run stays practical.

The design goal is not to keep learning pseudoGT forever.  Phase 1
creates stable client/class/domain differences for DQA, and Phase 2
briefly lets that target signal reach the full detector.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "dynamic_quality_aware_classwise_aggregation").exists():
    PROJECT_ROOT = cwd / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa"
else:
    PROJECT_ROOT = cwd

WORKSPACE = PROJECT_ROOT / "output" / "02_head_to_full_long_dqa"
RUNNER = PROJECT_ROOT / "scripts" / "run_scene_daynight_dqa_02_head_to_full.py"

print("PROJECT_ROOT", PROJECT_ROOT)
print("WORKSPACE", WORKSPACE)
print("RUNNER", RUNNER)

PROJECT_ROOT /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa
WORKSPACE /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa
RUNNER /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_02_head_to_full.py


## Schedule And Runtime Estimate

In [2]:
PHASE1_ROUNDS = 30
PHASE2_ROUNDS = 2
PHASE1_MIN_PER_ROUND = 19.0
PHASE2_MIN_PER_ROUND = 23.0
FINAL_EVAL_MIN = 60.0

estimate = pd.DataFrame([
    {
        "stage": "Phase1 head-only",
        "rounds": PHASE1_ROUNDS,
        "minutes_per_round": PHASE1_MIN_PER_ROUND,
        "estimated_minutes": PHASE1_ROUNDS * PHASE1_MIN_PER_ROUND,
    },
    {
        "stage": "Phase2 full burst",
        "rounds": PHASE2_ROUNDS,
        "minutes_per_round": PHASE2_MIN_PER_ROUND,
        "estimated_minutes": PHASE2_ROUNDS * PHASE2_MIN_PER_ROUND,
    },
    {
        "stage": "final-focused paper eval",
        "rounds": 0,
        "minutes_per_round": None,
        "estimated_minutes": FINAL_EVAL_MIN,
    },
])
total_minutes = float(estimate["estimated_minutes"].sum())
display(estimate)
print(f"Estimated total: {total_minutes / 60:.2f} hours ({total_minutes:.0f} minutes)")

,stage,rounds,minutes_per_round,estimated_minutes
0,Phase1 head-only,30,19.0,570.0
1,Phase2 full burst,2,23.0,46.0
2,final-focused paper eval,0,NaN,60.0


Estimated total: 11.27 hours (676 minutes)


## Runner Defaults

In [3]:
spec = importlib.util.spec_from_file_location("run_scene_daynight_dqa_02_head_to_full", RUNNER)
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)

args = runner.parse_args([])
phase1 = runner.default_phase1_spec(args)
phase2 = runner.default_phase2_spec(args)
display(pd.DataFrame([
    {"phase": "phase1", **runner.asdict(phase1)},
    {"phase": "phase2", **runner.asdict(phase2)},
]))

,phase,name,display_name,note,train_scope,aggregate_scope,client_lr0,source_repeat,pseudo_repeat,orthogonal_weight,loss_box,dqa_min_server_alpha,dqa_server_anchor,dqa_residual_blend,dqa_classwise_blend
0,phase1,phase1_head,Phase 1 head-only DQA,Long head/neck-only source-anchored pseudoGT a...,neck_head,all,0.0008,1,2,0.0001,0.005,0.70,10.0,0.14,0.16
1,phase2,phase2_full,Phase 2 full-model burst DQA,Short full-model low-LR target burst. The goal...,all,all,0.0003,1,1,0.0001,0.010,0.76,14.0,0.10,0.12


## Setup Only

In [4]:
subprocess.run([
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--client-limit", "1500",
    "--phase1-rounds", str(PHASE1_ROUNDS),
    "--phase2-rounds", str(PHASE2_ROUNDS),
    "--setup-only",
], cwd=PROJECT_ROOT, check=True)

{
  "protocol": "scene_daynight_dqa_02_head_to_full_long_v1",
  "workspace": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa",
  "clients": [
    {
      "id": 0,
      "name": "highway_day",
      "weather": "highway_day",
      "scene": "highway",
      "timeofday": "daytime"
    },
    {
      "id": 1,
      "name": "highway_night",
      "weather": "highway_night",
      "scene": "highway",
      "timeofday": "night"
    },
    {
      "id": 2,
      "name": "citystreet_day",
      "weather": "citystreet_day",
      "scene": "city street",
      "timeofday": "daytime"
    },
    {
      "id": 3,
      "name": "citystreet_night",
      "weather": "citystreet_night",
      "scene": "city street",
      "timeofday": "night"
    },
    {
      "id": 4,
      "name": "residential_day",
      "weather": "residential_day",
      "scene": "residential",
      "timeofday": "daytime"
    },
    {
      "id": 5,
      "name

CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_02_head_to_full.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa', '--client-limit', '1500', '--phase1-rounds', '30', '--phase2-rounds', '2', '--setup-only'], returncode=0)

## Run Phase1 30 / Phase2 2

The runner uses `tqdm` internally and writes live progress to:

```text
output/02_head_to_full_long_dqa/stats/02_head_to_full_progress.csv
```

Default evaluation is final-focused: warmup, Phase1 final aggregate,
Phase1 final repair, Phase2 final aggregate, and Phase2 final repair.

In [5]:
CLIENT_LIMIT = 1500
BATCH_SIZE = 160
WORKERS = 8
GPUS = 2
DEVICE = ""
MAX_IMAGES_PER_CLIENT = 0

cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--client-limit", str(CLIENT_LIMIT),
    "--phase1-rounds", str(PHASE1_ROUNDS),
    "--phase2-rounds", str(PHASE2_ROUNDS),
    "--batch-size", str(BATCH_SIZE),
    "--workers", str(WORKERS),
    "--gpus", str(GPUS),
    "--device", DEVICE,
    "--master-port", "31141",
    "--max-images-per-client", str(MAX_IMAGES_PER_CLIENT),
    "--estimated-phase1-round-minutes", str(PHASE1_MIN_PER_ROUND),
    "--estimated-phase2-round-minutes", str(PHASE2_MIN_PER_ROUND),
    "--estimated-eval-minutes", str(FINAL_EVAL_MIN),
    "--evaluate",
    "--classwise",
    "--no-eval-plots",
    "--notify",
]

print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_02_head_to_full.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa --client-limit 1500 --phase1-rounds 30 --phase2-rounds 2 --batch-size 160 --workers 8 --gpus 2 --device  --master-port 31141 --max-images-per-client 0 --estimated-phase1-round-minutes 19.0 --estimated-phase2-round-minutes 23.0 --estimated-eval-minutes 60.0 --evaluate --classwise --no-eval-plots --notify
DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)
{
  "protocol": "scene_daynight_dqa_02_head_to_full_long_v1",
  "workspace": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa",
  "clients": [
    {
      "id": 0,
      "name": "highway_day",
      "weather": "highway_day

02 head-to-full DQA:   0%|          | 0/32 [00:00<?, ?round/s]EfficientTeacher  618670a torch 2.8.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48508.9375MB)

Fusing layers... 
Model summary: 365 layers, 46156743 parameters, 0 gradients
/root/micromamba/envs/al_yolov8/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


round001 client0_highway_day: pseudo scan 1/1500 images, kept 12 boxes
round001 client0_highway_day: pseudo scan 250/1500 images, kept 2318 boxes
round001 client0_highway_day: pseudo scan 500/1500 images, kept 4696 boxes
round001 client0_highway_day: pseudo scan 750/1500 images, kept 7005 boxes
round001 client0_highway_day: pseudo scan 1000/1500 images, kept 9304 boxes
round001 client0_highway_day: pseudo scan 1250/1500 images, kept 11580 boxes
round001 client0_highway_day: pseudo scan 1500/1500 images, kept 13881 boxes
{
  "round": "round001",
  "client": "client0_highway_day",
  "teacher": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa/global_checkpoints/round000_warmup.pt",
  "source_images_scanned": 1500,
  "pseudo_images_kept": 1488,
  "pseudo_boxes_kept": 10812,
  "boxes_per_kept_image": 7.266129032258065,
  "mean_conf": 0.7457716877548465,
  "mean_stability": 0.9449824729243165,
  "mean_score": 0.707788491927


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


train: __immutable__=False, __deprecated_keys__=set(), __renamed_keys__={}, __new_allowed__=False


...

Results saved to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa/runs/pl03_round032_phase2_full_server_repair

1 epochs completed in 0.028 hours.
Destroying process group... 
[rank0]:[W507 21:45:33.902225788 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Cleaned 3 training artifact(s), freed 0.26 GiB
/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/evaluate_scene_daynight_protocol.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa --splits highway_day,highway_night,citystreet_day,citystreet_night,residential_day,residential_night,total --batch-size 16 --no-plots --verbose --checkpoint warmup_global=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa/global_checkpoints/round000_warmup.pt --checkpoint phase1_final_aggregate=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa/checkpoints/round030_phase1_head_dqa_aggregate.pt --checkpoint phase1_final_repair=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/outpu

02 head-to-full DQA: 100%|██████████| 32/32 [7:38:15<00:00, 859.23s/round, elapsed=7h38m15s, eta=0s, phase=phase2_full, round=2]    


Saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa/validation_reports/paper_protocol_eval_manifest.json
Saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa/validation_reports/paper_protocol_eval_summary.csv
Saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa/validation_reports/paper_protocol_classwise_summary.csv
Saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa/validation_reports/paper_protocol_eval_summary.md
Saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa/stats/02_head_to_full_final_metrics.csv
DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)


CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_02_head_to_full.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/02_head_to_full_long_dqa', '--client-limit', '1500', '--phase1-rounds', '30', '--phase2-rounds', '2', '--batch-size', '160', '--workers', '8', '--gpus', '2', '--device', '', '--master-port', '31141', '--max-images-per-client', '0', '--estimated-phase1-round-minutes', '19.0', '--estimated-phase2-round-minutes', '23.0', '--estimated-eval-minutes', '60.0', '--evaluate', '--classwise', '--no-eval-plots', '--notify'], returncode=0)

## Progress

In [6]:
progress_csv = WORKSPACE / "stats" / "02_head_to_full_progress.csv"
if progress_csv.exists():
    progress_df = pd.read_csv(progress_csv)
    display(progress_df.tail(10))
    latest = progress_df.tail(1).iloc[0]
    print(f"elapsed={latest['elapsed_hms']} eta={latest['eta_hms']} completed={latest['completed_rounds']}/{latest['total_rounds']}")
else:
    print("No progress CSV yet:", progress_csv)

,created_utc,phase,phase_round,global_round,completed_rounds,total_rounds,elapsed_seconds,avg_seconds_per_round,eta_seconds,elapsed_hms,eta_hms,checkpoint
22,2026-05-07T19:32:29.914805+00:00,phase1_head,23,23,23,32,19508.704,848.205,7633.841,5h25m08s,2h07m13s,/app/Object_Detection/dynamic_quality_aware_cl...
23,2026-05-07T19:46:28.714540+00:00,phase1_head,24,24,24,32,20347.504,847.813,6782.501,5h39m07s,1h53m02s,/app/Object_Detection/dynamic_quality_aware_cl...
24,2026-05-07T20:00:41.739039+00:00,phase1_head,25,25,25,32,21200.529,848.021,5936.148,5h53m20s,1h38m56s,/app/Object_Detection/dynamic_quality_aware_cl...
25,2026-05-07T20:14:43.248973+00:00,phase1_head,26,26,26,32,22042.039,847.771,5086.624,6h07m22s,1h24m46s,/app/Object_Detection/dynamic_quality_aware_cl...
26,2026-05-07T20:28:50.911590+00:00,phase1_head,27,27,27,32,22889.701,847.767,4238.834,6h21m29s,1h10m38s,/app/Object_Detection/dynamic_quality_aware_cl...
27,2026-05-07T20:42:53.172144+00:00,phase1_head,28,28,28,32,23731.962,847.570,3390.280,6h35m31s,56m30s,/app/Object_Detection/dynamic_quality_aware_cl...
28,2026-05-07T20:57:01.195053+00:00,phase1_head,29,29,29,32,24579.985,847.586,2542.757,6h49m39s,42m22s,/app/Object_Detection/dynamic_quality_aware_cl...
29,2026-05-07T21:11:10.229287+00:00,phase1_head,30,30,30,32,25429.019,847.634,1695.268,7h03m49s,28m15s,/app/Object_Detection/dynamic_quality_aware_cl...
30,2026-05-07T21:28:28.060720+00:00,phase2_full,1,31,31,32,26466.850,853.769,853.769,7h21m06s,14m13s,/app/Object_Detection/dynamic_quality_aware_cl...
31,2026-05-07T21:45:36.580847+00:00,phase2_full,2,32,32,32,27495.370,859.230,0.000,7h38m15s,0s,/app/Object_Detection/dynamic_quality_aware_cl...


elapsed=7h38m15s eta=0s completed=32/32


## Final Metrics

In [7]:
final_metrics = WORKSPACE / "stats" / "02_head_to_full_final_metrics.csv"
if final_metrics.exists():
    final_df = pd.read_csv(final_metrics)
    display(final_df)
else:
    print("No final metrics yet:", final_metrics)

,checkpoint_label,kind,phase,phase_round,global_round,precision,recall,map50,map50_95,gain_vs_warmup_map50_95,delta_vs_repair_only_r3_map50_95,worst_split,worst_split_map50_95,worst_delta_vs_repair_only_r3_map50_95,day_avg_map50_95,night_avg_map50_95,night_delta_vs_repair_only_r3_map50_95,day_night_gap_map50_95
0,warmup_global,warmup,warmup,NaN,NaN,0.531,0.336,0.350,0.188,0.000,-0.024,highway_night,0.130,-0.022,0.208000,0.150000,-0.023333,0.058000
1,phase1_final_aggregate,aggregate,phase1_head,30.0,30.0,0.519,0.368,0.350,0.190,0.002,-0.022,highway_night,0.121,-0.031,0.214000,0.140667,-0.032666,0.073333
2,phase1_final_repair,server_repair,phase1_head,30.0,30.0,0.505,0.369,0.345,0.187,-0.001,-0.025,highway_night,0.120,-0.032,0.210000,0.137667,-0.035666,0.072333
3,phase2_final_aggregate,aggregate,phase2_full,2.0,32.0,0.515,0.364,0.345,0.186,-0.002,-0.026,highway_night,0.117,-0.035,0.210333,0.134333,-0.039000,0.076000
4,phase2_final_repair,server_repair,phase2_full,2.0,32.0,0.507,0.367,0.342,0.185,-0.003,-0.027,highway_night,0.118,-0.034,0.208000,0.133333,-0.040000,0.074667


## Split-Level Final Evaluation

In [8]:
split_metrics = WORKSPACE / "stats" / "02_head_to_full_split_metrics.csv"
if split_metrics.exists():
    split_df = pd.read_csv(split_metrics)
    cols = ["checkpoint_label", "split", "images", "labels", "precision", "recall", "map50", "map50_95"]
    display(split_df[cols].sort_values(["split", "checkpoint_label"]))
else:
    print("No split metrics yet:", split_metrics)

,checkpoint_label,split,images,labels,precision,recall,map50,map50_95
9,phase1_final_aggregate,citystreet_day,3067.0,69855.0,0.532,0.409,0.393,0.217
16,phase1_final_repair,citystreet_day,3067.0,69855.0,0.523,0.406,0.388,0.214
23,phase2_final_aggregate,citystreet_day,3067.0,69855.0,0.527,0.407,0.389,0.214
30,phase2_final_repair,citystreet_day,3067.0,69855.0,0.524,0.404,0.385,0.211
2,warmup_global,citystreet_day,3067.0,69855.0,0.558,0.336,0.383,0.209
10,phase1_final_aggregate,citystreet_night,2582.0,47259.0,0.471,0.317,0.278,0.143
17,phase1_final_repair,citystreet_night,2582.0,47259.0,0.464,0.310,0.273,0.140
24,phase2_final_aggregate,citystreet_night,2582.0,47259.0,0.460,0.315,0.270,0.138
31,phase2_final_repair,citystreet_night,2582.0,47259.0,0.475,0.309,0.269,0.137
3,warmup_global,citystreet_night,2582.0,47259.0,0.462,0.314,0.293,0.149


## Compare Against 01_0 Repair-Only Summary

In [9]:
baseline = PROJECT_ROOT / "output" / "01_0_repair_baseline_comparison" / "stats" / "01_0_all_condition_metrics.csv"
if final_metrics.exists() and baseline.exists():
    base = pd.read_csv(baseline)
    repair_only = base[base["condition"].eq("repair_only")].sort_values("round").tail(1)
    display(repair_only)
    cols = [
        "checkpoint_label",
        "map50",
        "map50_95",
        "delta_vs_repair_only_r3_map50_95",
        "worst_split",
        "worst_split_map50_95",
        "worst_delta_vs_repair_only_r3_map50_95",
        "night_avg_map50_95",
        "night_delta_vs_repair_only_r3_map50_95",
    ]
    display(final_df[cols])
else:
    print("Missing final metrics or baseline.")

,condition,round,aggregate_map50,aggregate_map50_95,repaired_map50,repaired_map50_95,repair_gain_map50_95,retained_gain_map50_95,round_delta_map50_95,worst_split,worst_split_map50_95,day_avg_map50_95,night_avg_map50_95,day_night_gap_map50_95
2,repair_only,3,NaN,NaN,0.381,0.212,NaN,0.024,0.002,highway_night,0.152,0.232333,0.173333,0.059


,checkpoint_label,map50,map50_95,delta_vs_repair_only_r3_map50_95,worst_split,worst_split_map50_95,worst_delta_vs_repair_only_r3_map50_95,night_avg_map50_95,night_delta_vs_repair_only_r3_map50_95
0,warmup_global,0.350,0.188,-0.024,highway_night,0.130,-0.022,0.150000,-0.023333
1,phase1_final_aggregate,0.350,0.190,-0.022,highway_night,0.121,-0.031,0.140667,-0.032666
2,phase1_final_repair,0.345,0.187,-0.025,highway_night,0.120,-0.032,0.137667,-0.035666
3,phase2_final_aggregate,0.345,0.186,-0.026,highway_night,0.117,-0.035,0.134333,-0.039000
4,phase2_final_repair,0.342,0.185,-0.027,highway_night,0.118,-0.034,0.133333,-0.040000


## PseudoGT Signal Trend

In [10]:
pseudo_rows = []
stats_dir = WORKSPACE / "stats"
if stats_dir.exists():
    for path in sorted(stats_dir.glob("03_round*_pseudo_label_stats.csv")):
        part = pd.read_csv(path)
        part["source_csv"] = path.name
        pseudo_rows.append(part)
if pseudo_rows:
    pseudo_df = pd.concat(pseudo_rows, ignore_index=True)
    summary_cols = [c for c in [
        "round", "client", "images", "kept_images", "boxes",
        "mean_conf", "mean_stability", "mean_score", "source_csv",
    ] if c in pseudo_df.columns]
    display(pseudo_df[summary_cols].tail(30))
else:
    print("No pseudoGT stats yet.")

,round,client,mean_conf,mean_stability,mean_score,source_csv
162,round028,client0_highway_day,0.842352,0.941735,0.796607,03_round028_pseudo_label_stats.csv
163,round028,client1_highway_night,0.762103,0.920629,0.705922,03_round028_pseudo_label_stats.csv
164,round028,client2_citystreet_day,0.842357,0.939176,0.794684,03_round028_pseudo_label_stats.csv
165,round028,client3_citystreet_night,0.752466,0.914847,0.693096,03_round028_pseudo_label_stats.csv
166,round028,client4_residential_day,0.872944,0.949169,0.832053,03_round028_pseudo_label_stats.csv
167,round028,client5_residential_night,0.808280,0.930653,0.755730,03_round028_pseudo_label_stats.csv
168,round029,client0_highway_day,0.844458,0.941270,0.798143,03_round029_pseudo_label_stats.csv
169,round029,client1_highway_night,0.765653,0.919778,0.708569,03_round029_pseudo_label_stats.csv
170,round029,client2_citystreet_day,0.844293,0.938592,0.796038,03_round029_pseudo_label_stats.csv
171,round029,client3_citystreet_night,0.755830,0.913908,0.695483,03_round029_pseudo_label_stats.csv
